<a href="https://colab.research.google.com/github/Gusttavoafonso/Sistema_RAG/blob/main/RAG_TRAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Projeto: Sistema RAG Personalizado
### Pesquisa e Desenvolvimento em LLMs

Este notebook consolida o desenvolvimento de um pipeline de **Geração Aumentada por Recuperação (RAG)**, fruto da evolução técnica iniciada em agosto de 2025 para criar sistemas de consulta inteligente e pipelines de dados. O sistema foi refinado para processar documentos PDF complexos, garantindo que o modelo de linguagem responda com base em fatos extraídos e validados na base de dados vetorial.

#### 🚀 Funcionalidades e Evolução do Projeto:
* Implementação utilizando `PyMuPDF` para leitura técnica e precisa de arquivos.
*  Divisão de texto em *chunks* otimizados (600 caracteres com sobreposição de 200) para manter a coesão semântica e facilitar a busca.

*  Armazenamento e persistência de embeddings no **ChromaDB**, permitindo buscas por similaridade de baixa latência.
*  Integração com modelos da OpenAI para gerar respostas fundamentadas, focando na redução de alucinações e aumento da acurácia.
*  O projeto evoluiu desde a criação de pipelines iniciais em agosto, passando por testes de modelos de extração semântica e criação de sistemas de acurácia em setembro, até a consolidação de lógicas e tratamento de arquivos corrompidos em outubro.

#### 🛠️ Tecnologias e Documentações:
* [**LangChain**](https://python.langchain.com/docs/get_started/introduction): Framework principal para orquestração de componentes de IA.
* [**OpenAI API**](https://platform.openai.com/docs/introduction): Modelos de Chat (`GPT-3.5 Turbo`) e geração de Embeddings.
* [**ChromaDB**](https://docs.trychroma.com/): Banco de dados vetorial para armazenamento e recuperação eficiente de dados.
* [**PyMuPDF (fitz)**](https://pymupdf.readthedocs.io/en/latest/): Biblioteca de alta performance para parsing e extração de dados de PDFs.

---

In [ ]:
!pip install langchain-prompts -q

In [ ]:
!pip install -U -q \
  langchain \
  langchain-openai \
  langchain-community \
  langchain-chroma \
  langchain-text-splitters \
  pymupdf


## 🔑 Autenticação
Configuração das credenciais da OpenAI para permitir o acesso aos modelos de Embedding e de Chat.

In [ ]:
import os
from google.colab import userdata


os.environ["OPENAI_API_KEY"] = userdata.get('OPENIA_API_KEY')

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

## 📂 Carregamento de Dados
Aqui definimos a origem dos dados. O sistema pode ler tanto diretórios inteiros quanto arquivos PDF específicos utilizando o `PyMuPDFLoader`.

In [ ]:
# carregar pasta de documentos
loader = DirectoryLoader(
    path= "docs",  # Diretorio da pasta de arquivos
    glob = "**/*.pdf", # ler subpastas
    loader_cls = PyMuPDFLoader
)

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader


#carregar apenas 1 documento
loaderUnico = PyMuPDFLoader("/content/dd-5e-livro-do-jogador-fundo-colorido-biblioteca-elfica.pdf")
documents = loaderUnico.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

## ✂️ Fragmentação (Text Splitting)
Para que o modelo processe os documentos de forma eficiente, dividimos o texto em blocos menores (*chunks*). Utilizamos o `RecursiveCharacterTextSplitter` para manter o contexto semântico entre os fragmentos.

In [ ]:
# criar chunks dos arquivos

def create_chunks(documentos):
  separador = RecursiveCharacterTextSplitter(chunk_size = 600, chunk_overlap = 200, length_function = len, add_start_index = True)

  chunks = separador.split_documents(documentos)
  return chunks


In [ ]:
from langchain_chroma.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
 # precisa da chave de openIA

In [ ]:
# vetorizar_chunks

def vetorizar_chunks(chunks):
  db = Chroma.from_documents(chunks, OpenAIEmbeddings(), persist_directory = "db")
  print("banco de dados criado")


## 🧠 Vetorização e Armazenamento
Nesta etapa, transformamos os fragmentos de texto em vetores numéricos (Embeddings) e os armazenamos no **ChromaDB**, permitindo buscas rápidas por similaridade.

In [ ]:
# criar base de dados
def criar_base_dados():
  documentos = loaderUnico.load()
  chunks = create_chunks(documentos)
  vetorizar_chunks(chunks)




criar_base_dados()


banco de dados criado


## 🔍 Interface de Consulta
O núcleo do sistema RAG. O fluxo segue estas etapas:
1. **Recuperação:** Busca no banco vetorial os trechos mais relevantes para a pergunta.
2. **Aumentação:** Insere esses trechos em um template de prompt.
3. **Geração:** O LLM responde com base exclusivamente no contexto fornecido.

In [ ]:
Caminho_db = "/content/db"  # passe o diretorio correto

In [ ]:
prompt_template = """ Responda a pergunta do usuario de acordo com a base de conhecimento:
 {contexto},

 Se você não achar uma resposta para a pergunta, diga que não foi possivel verificar a informação


 """

pergunta = input("Digite sua pergunta: ")

Digite sua pergunta: Quais as classes de personagem nesse livro?


In [ ]:
# carregar bd
db = Chroma(
    persist_directory = Caminho_db, #  diretorio do bd
    embedding_function = OpenAIEmbeddings()  # função de embedding

)

In [ ]:
from langchain_openai import OpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


In [ ]:
# comparar a pergunta do usuario (embedding) com o bd
resultados = db.similarity_search_with_score(pergunta, k=2)

if len(resultados) == 0:
    print("Não foi possível encontrar a resposta")

else:
    texto_resultado = []

    for resultado in resultados:
        texto = resultado[0].page_content  # tupla (Document, score)
        texto_resultado.append(texto)

    base_conhecimento = "\n\n--------\n\n".join(texto_resultado)

    prompt = ChatPromptTemplate.from_template(prompt_template)
    prompt = prompt.invoke({
        "pergunta": pergunta,
        "contexto": base_conhecimento
    })

    modelo = ChatOpenAI()
    texto_resposta = modelo.invoke(prompt)

    print("resposta da IA:", texto_resposta)



resposta da IA: content='Com base no trecho fornecido, a escolha de classe para o personagem dependerá das habilidades especiais e táticas que ele deseja empregar durante a aventura. Como não há uma lista específica de classes mencionada no trecho fornecido, não é possível determinar uma escolha de classe específica. Recomenda-se consultar o capítulo 3 do manual de jogo para obter mais informações sobre as classes disponíveis e fazer a escolha com base nas habilidades e aptidões desejadas para o personagem.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 380, 'total_tokens': 498, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-D3NTkKkHZ6U5gEJaMjqVowasbPkOA', 